# 132STA — Direct stiffness method for a plane frame

Analyze the frame from Lecture 6 using manual code numbers, then a helper.

Replace each `...` at a TODO before running that cell. Use the lecture and handout to justify your choices.

### 0. Preparation

Run the compact **Setup** cell once, then run the imports below. Expand the setup code only if you want to inspect it.

In [ ]:
# @title Setup: load teaching functions (run first) { display-mode: "form" }
# Locate the teaching package (Colab or local Jupyter).
from pathlib import Path
import subprocess
import sys

repo = Path.cwd()
if not (repo / "sta132").is_dir():
    if (repo.parent / "sta132").is_dir():
        repo = repo.parent
    else:
        repo = Path("/content/132sta-python")
        if not (repo / "sta132").is_dir():
            subprocess.run(
                ["git", "clone", "--quiet", "--depth", "1",
                 "https://github.com/MartinDoskar/132STA-python.git", str(repo)],
                check=True,
            )
if not all((repo / "sta132" / name).is_file()
           for name in ["geometry.py", "beam.py", "assembly.py"]):
    raise FileNotFoundError("Use the course version containing geometry.py, beam.py and assembly.py.")
sys.path.insert(0, str(repo))
print("Teaching functions are ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sta132.beam import (
    stiffness_matrix_beam,
    fixed_end_forces_uniform_load_beam,
    fixed_end_forces_midpoint_load_beam,
    end_forces_beam,
)
from sta132.plotting import plot_frame

np.set_printoptions(precision=4, suppress=True, linewidth=120)

### 1. Define the problem

Define the joints, members, properties, loads, and prescribed $w_4=\Delta=10$ mm.

In [ ]:
# Joint coordinates [x, z] in metres
xz = np.array([[0., 0.],   # Joint 1
               [4., 0.],   # Joint 2
               [0., 3.],   # Joint 3
               [8., 3.]])  # Joint 4

# Endpoint joint numbers, STARTING AT ONE, in the member orientation.
member_node_numbers = np.array([[3, 1],
                                [1, 2],
                                [2, 4]])

E = 21e6                # kN/m^2
A = 0.3 * 0.4           # m^2
I = 0.3 * 0.4**3 / 12   # m^4
H = 40.                 # kN, global x
q = 15.                 # kN/m, downward on member 1--2
P = 60.                 # kN, global z at the midpoint of member 2--4
delta = 0.01            # m, prescribed downward motion of joint 4

### 2. Member stiffness relations

Compute the global element stiffness matrices and coordinate transformations.

In [ ]:
K1, _, _ = stiffness_matrix_beam(E, A, I, xz[2], xz[0])  # Member 3--1
K2, _, _ = stiffness_matrix_beam(E, A, I, xz[0], xz[1])  # Member 1--2
K3, _, T3 = stiffness_matrix_beam(E, A, I, xz[1], xz[3])  # Member 2--4

print("K1:\n", K1)
print("K2:\n", K2)
print("K3:\n", K3)

### 3. Member loading

Compute fixed-end actions in global and local coordinates. Resolve the
vertical midpoint force $P$ into local components using the inclined member's $T_3$.

In [ ]:
fixed_end_forces1_global = np.zeros(6)
fixed_end_forces1_local = np.zeros(6)
fixed_end_forces2_global, fixed_end_forces2_local, _ = fixed_end_forces_uniform_load_beam(
    0., q, xz[0], xz[1]
)

# TODO: Resolve the global vertical force into local [fx, fz] components.
force3_local = ...
if force3_local is Ellipsis:
    raise NotImplementedError("Complete the midpoint-force transformation.")
fx3, fz3 = force3_local

# TODO: Compute the midpoint fixed-end actions and T for member 2--4.
fixed_end_result3 = ...
if fixed_end_result3 is Ellipsis:
    raise NotImplementedError("Complete the midpoint fixed-end actions.")
fixed_end_forces3_global, fixed_end_forces3_local, _ = fixed_end_result3

print("Midpoint force on member 2--4, local [fx, fz] [kN]:", force3_local)
print("Local fixed-end actions [X_a, Z_a, M_a, X_b, Z_b, M_b]:")
print("Member 3--1:", fixed_end_forces1_local)
print("Member 1--2:", fixed_end_forces2_local)
print("Member 2--4:", fixed_end_forces3_local)
print("X, Z use kN; M uses kN m.")

### 4. Manually assigned code numbers

Assign one-based full code numbers in member endpoint order.
Subtract 1 to form separate Python indices.

In [ ]:
code_numbers1 = np.array([7, 8, 9, 1, 2, 3])      # Member 3--1
code_numbers2 = np.array([1, 2, 3, 4, 5, 6])      # Member 1--2
code_numbers3 = np.array([4, 5, 6, 10, 11, 12])   # Member 2--4

# Lecture code numbers start at 1; Python array indices start at 0.
indices1 = code_numbers1 - 1
indices2 = code_numbers2 - 1
indices3 = code_numbers3 - 1

print("Member 3--1 code numbers:", code_numbers1)
print("Member 3--1 Python indices:", indices1)

### 5. Assembly

Assemble the full stiffness matrix and right-hand side, including nodal
and equivalent member loads.

In [ ]:
K = np.zeros((12, 12))
K[np.ix_(indices1, indices1)] += K1
K[np.ix_(indices2, indices2)] += K2
K[np.ix_(indices3, indices3)] += K3

F_applied = np.zeros(12)
F_applied[0] = H  # Horizontal joint force at full code number 1.
F = F_applied.copy()
# Equivalent applied nodal loads oppose the global fixed-end actions.
F[indices1] -= fixed_end_forces1_global
F[indices2] -= fixed_end_forces2_global
F[indices3] -= fixed_end_forces3_global

print("Full assembled K:\n", K)
print("Assembled right-hand side F:", F)
print("K is symmetric:", np.allclose(K, K.T))

### 6. Supports and solution

Prescribe the support motions, including $w_4=\Delta$, and solve for the
unknown components. Retain the known-displacement contribution.

In [ ]:
unknown_code_numbers = np.array([1, 2, 3, 4, 5, 6])
prescribed_code_numbers = np.array([7, 8, 9, 10, 11, 12])
U = unknown_code_numbers - 1
C = prescribed_code_numbers - 1

d = np.zeros(12)
# Prescribed values at joints 3 and 4, in [u3, w3, phi3, u4, w4, phi4] order.
d[C] = np.array([0., 0., 0., 0., delta, 0.])
rhs = F[U] - K[np.ix_(U, C)] @ d[C]
d[U] = np.linalg.solve(K[np.ix_(U, U)], rhs)

print("Joint       u [mm]       w [mm]    phi [mrad]")
for joint, (u, w, phi) in enumerate(d.reshape(-1, 3), 1):
    print(f"{joint:5d}{1e3*u:13.5f}{1e3*w:13.5f}{1e3*phi:14.5f}")

### 7. Support reactions

Recover reactions from the full system and check force and moment equilibrium.

In [ ]:
reactions = K @ d - F
reaction_values = reactions.reshape(-1, 3)
print("Joint     Rx [kN]      Rz [kN]     RM [kN m]")
for joint in [3, 4]:
    Rx, Rz, RM = reaction_values[joint - 1]
    print(f"{joint:5d}{Rx:13.5f}{Rz:13.5f}{RM:14.5f}")
print("Unknown-component residuals (kN for forces, kN m for moments):", reactions[U])

# TODO: Check equilibrium of the whole frame using a free-body diagram.
balance_x = ...
balance_z = ...
balance_moment = ...  # About joint 1, positive counterclockwise.
if any(value is Ellipsis for value in [balance_x, balance_z, balance_moment]):
    raise NotImplementedError("Complete the whole-frame equilibrium checks.")
print("Horizontal force balance [kN]:", balance_x)
print("Vertical force balance [kN]:", balance_z)
print("Counterclockwise moment balance about joint 1 [kN m]:", balance_moment)

### 8. Member end actions

Recover global and local member end actions, including the fixed-end contributions.
Check force and moment equilibrium of joint 2 using a free-body diagram.

In [ ]:
de1, de2, de3 = d[indices1], d[indices2], d[indices3]
end_forces1_global, end_forces1_local, _ = end_forces_beam(
    E, A, I, xz[2], xz[0], de1, fixed_end_forces1_local
)
end_forces2_global, end_forces2_local, _ = end_forces_beam(
    E, A, I, xz[0], xz[1], de2, fixed_end_forces2_local
)
end_forces3_global, end_forces3_local, _ = end_forces_beam(
    E, A, I, xz[1], xz[3], de3, fixed_end_forces3_local
)

print("Global end actions [Xa, Za, Ma, Xb, Zb, Mb]:")
print("Member 3--1:", end_forces1_global)
print("Member 1--2:", end_forces2_global)
print("Member 2--4:", end_forces3_global)
print("Local end actions [Xa, Za, Ma, Xb, Zb, Mb]:")
print("Member 3--1:", end_forces1_local)
print("Member 1--2:", end_forces2_local)
print("Member 2--4:", end_forces3_local)
print("X, Z use kN; M uses kN m.")

# TODO: Form the three equilibrium residuals for joint 2.
joint_residual = ...
if joint_residual is Ellipsis:
    raise NotImplementedError("Complete the joint-equilibrium check.")
print("Joint 2 residuals [X, Z, M] (kN, kN, kN m):", joint_residual)

**Discussion: from trusses to beams.** In notebook 05, `axial_force_truss` returns one scalar per member. What replaces this result for a beam? How do member loads enter force recovery, and how are member end actions related to internal forces at a section? Discuss how you would construct the internal-force diagrams.

### 9. Interpolated deformed shape

Plot the nodal solution using linear axial and cubic Hermite transverse interpolation.
Deformation caused by member loading with zero endpoint motion is omitted; a member with all endpoint motions zero is plotted straight.

In [ ]:
# The plotting function takes zero-based node indices.
connectivity = member_node_numbers - 1
fig, ax = plot_frame(xz, connectivity, d, scale=100)
plt.show()

### 10. Stationary supports

Solve a second case with stationary supports ($\Delta=0$) and compare the displacements and reactions.

In [ ]:
d_stationary = np.zeros(12)
d_stationary[U] = np.linalg.solve(K[np.ix_(U, U)], F[U])
reactions_stationary = K @ d_stationary - F

print("Joint     w, stationary [mm]      w, settlement [mm]")
for joint in range(1, 5):
    index_w = 3 * (joint - 1) + 1
    print(f"{joint:5d}{1e3*d_stationary[index_w]:23.5f}{1e3*d[index_w]:24.5f}")
print("Stationary-support reactions at joints 3 and 4:\n",
      reactions_stationary.reshape(-1, 3)[[2, 3]])
reaction_increment = (reactions - reactions_stationary).reshape(-1, 3)
print("Settlement-only reaction increments:\n", reaction_increment[[2, 3]])

### 11. Code numbers from node numbers

Generate the same full code numbers from the endpoint node numbers.
Preserve endpoint order; subtract 1 for Python indices.

In [ ]:
from sta132.assembly import code_numbers_from_nodes

code_numbers1_helper = code_numbers_from_nodes([3, 1], dofs_per_node=3)
code_numbers2_helper = code_numbers_from_nodes([1, 2], dofs_per_node=3)
code_numbers3_helper = code_numbers_from_nodes([2, 4], dofs_per_node=3)

print("Member 3--1:", code_numbers1_helper,
      "matches manual:", np.array_equal(code_numbers1_helper, code_numbers1))
print("Member 1--2:", code_numbers2_helper,
      "matches manual:", np.array_equal(code_numbers2_helper, code_numbers2))
print("Member 2--4:", code_numbers3_helper,
      "matches manual:", np.array_equal(code_numbers3_helper, code_numbers3))

Repeat the assembly with the helper and compare with the manual result.

In [ ]:
K_helper = np.zeros_like(K)
F_helper = F_applied.copy()
member_matrices = [K1, K2, K3]
member_fixed_end_forces_global = [fixed_end_forces1_global, fixed_end_forces2_global, fixed_end_forces3_global]

for node_numbers, Ke, fixed_end_forces_global in zip(
        member_node_numbers, member_matrices, member_fixed_end_forces_global):
    code_numbers = code_numbers_from_nodes(node_numbers, dofs_per_node=3)
    indices = code_numbers - 1
    K_helper[np.ix_(indices, indices)] += Ke
    F_helper[indices] -= fixed_end_forces_global

unknown_code_numbers_helper = code_numbers_from_nodes([1, 2], dofs_per_node=3)
prescribed_code_numbers_helper = code_numbers_from_nodes([3, 4], dofs_per_node=3)
U_helper = unknown_code_numbers_helper - 1
C_helper = prescribed_code_numbers_helper - 1
d_helper = np.zeros_like(d)
d_helper[C_helper] = d[C]
rhs_helper = (F_helper[U_helper]
              - K_helper[np.ix_(U_helper, C_helper)] @ d_helper[C_helper])
d_helper[U_helper] = np.linalg.solve(K_helper[np.ix_(U_helper, U_helper)], rhs_helper)
reactions_helper = K_helper @ d_helper - F_helper

print("Same assembled stiffness:", np.allclose(K_helper, K))
print("Same assembled loading:", np.allclose(F_helper, F))
print("Same full displacement vector:", np.allclose(d_helper, d))
print("Same reaction vector:", np.allclose(reactions_helper, reactions))